# ⚙️ 02 - Data Preprocessing & Feature Engineering

Thực hiện tiền xử lý dữ liệu và trích xuất đặc trưng số học theo yêu cầu đề bài:
- **1. Làm sạch văn bản**: Loại bỏ stop words, punctuation, HTML tags/URLs, chuẩn hóa Unicode
- **2. Phân chia tập dữ liệu độc lập** (Early Stratified Split 80/10/10) chống rò rỉ dữ liệu
- **3. Vector hóa văn bản bằng TF-IDF** (TfidfVectorizerScratch - Unigram & Bigram)
- **4. Trích xuất đặc trưng hình thức** (Tần suất ký tự '!', '$', chữ in hoa, độ dài)
- **5. Ghép nối không gian ma trận lai** (Hybrid Feature Matrix: Word + Char + Keywords + Numerics)
- **6. Thử nghiệm các tổ hợp đặc trưng** (Feature Combinations Experiment)


Notebook này **không gọi trực tiếp các model có sẵn** như `sklearn.linear_model`,
`sklearn.svm`, `sklearn.naive_bayes`, `sklearn.ensemble`, `sklearn.model_selection`
hoặc `sklearn.feature_extraction`.

Các thuật toán được gọi từ package tự xây dựng của project:

```text
src/
├── data/
├── features/
├── models/
├── evaluation/
└── deployment/
```

Các thư viện tính toán nền tảng (NumPy/Pandas/Scipy/Matplotlib...) nếu có được sử dụng
thì được **ẩn bên trong `src/`**, còn notebook chỉ làm việc với API của project.

## 1. Import Thư Viện & Cấu Hình Môi Trường
- Thiết lập đường dẫn động `PROJECT_ROOT` từ `Path.cwd()`.
- Nạp các module tự xây dựng từ `src.data` và `src.features`.


In [6]:
import os
import sys
import pandas as pd
import numpy as np
from IPython.display import display
from pathlib import Path

# Thiết lập đường dẫn gốc tương đối an toàn
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SpamExperimentConfig
from src.data.loader import load_raw_data
from src.data.preprocessing import (
    clean_email_split,
    stratified_split_dataframe,
    stratified_split_records,
    SpamTextProcessor,
    audit_data_quality,
)
from src.features.feature_engineering import (
    CSRMatrixScratch,
    TfidfVectorizerScratch,
    MaxAbsScalerScratch,
    SpamSignalFeatureExtractor,
    HybridFeatureBuilderScratch,
    experiment_feature_combinations,
    export_processed_data,
)

print("Project root:", PROJECT_ROOT)
print("Nạp các module Task 1 thành công!")


Project root: E:\Hoc_May_Tren_truong\lab_1 spam_classifier
Nạp các module Task 1 thành công!


## 2. Làm Sạch Toàn Bộ Dữ Liệu & Phân Tầng Độc Lập (Early Stratified Split)
- **Quy tắc chống rò rỉ dữ liệu (Data Leakage)**:
  1. Gọi `clean_email_split()` trên TOÀN BỘ tập dữ liệu thô để loại bỏ dòng rỗng, nhãn không hợp lệ, và xử lý mâu thuẫn nhãn/duplicate chéo.
  2. Phân tầng sớm 80% Train - 10% Validation - 10% Test bằng `stratified_split_records()` với `random_state=42`.


In [7]:
# 1. Nạp dữ liệu thô dạng DataFrame
df_raw = load_raw_data(PROJECT_ROOT / "data" / "raw" / "spam.csv", as_dataframe=True)
print(f"Tổng số bản ghi thô: {len(df_raw):,}")

# 2. Báo cáo kiểm toán chất lượng dữ liệu (Audit DataFrame)
quality_report = audit_data_quality(df_raw)
df_audit = pd.DataFrame([
    {"Chỉ tiêu kiểm định": "Tổng số dòng đầu vào (Input Rows)", "Số lượng": f"{quality_report['input_rows']:,}"},
    {"Chỉ tiêu kiểm định": "Dòng nhãn không hợp lệ (Invalid Label)", "Số lượng": f"{quality_report['invalid_label_rows']:,}"},
    {"Chỉ tiêu kiểm định": "Dòng văn bản rỗng (Empty Text)", "Số lượng": f"{quality_report['empty_text_rows']:,}"},
    {"Chỉ tiêu kiểm định": "Nhóm mâu thuẫn nhãn (Conflicting Groups)", "Số lượng": f"{quality_report['conflicting_groups']:,}"},
    {"Chỉ tiêu kiểm định": "Số dòng mâu thuẫn nhãn (Conflicting Rows)", "Số lượng": f"{quality_report['conflicting_rows']:,}"},
    {"Chỉ tiêu kiểm định": "Dòng trùng lặp loại bỏ (Duplicates Removed)", "Số lượng": f"{quality_report['duplicate_rows_removed']:,}"},
    {"Chỉ tiêu kiểm định": "Dòng sạch được giữ lại (Retained Clean Rows)", "Số lượng": f"{quality_report['retained_rows']:,}"},
])
display(df_audit)

# 3. Làm sạch dữ liệu toàn diện trước khi chia tập
df_clean = clean_email_split(df_raw, as_dataframe=True)
print(f"Số bản ghi sạch giữ lại: {len(df_clean):,}")

# 4. Phân tầng độc lập 80/10/10 chống rò rỉ dữ liệu
train_df, val_df, test_df = stratified_split_dataframe(
    df_clean,
    label_col="label",
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
    random_state=42,
)

# 5. Bảng thống kê phân phối các tập phân tầng (Stratified Split Summary DataFrame)
split_summary = []
for name, split_data in [("Train (80%)", train_df), ("Val (10%)", val_df), ("Test (10%)", test_df)]:
    counts = split_data["label"].value_counts()
    n_total = len(split_data)
    split_summary.append({
        "Tập phân chia": name,
        "Tổng số mẫu": f"{n_total:,}",
        "Số lượng HAM": f"{counts.get('ham', 0):,}",
        "Tỷ lệ HAM (%)": f"{(counts.get('ham', 0) / n_total * 100):.2f}%",
        "Số lượng SPAM": f"{counts.get('spam', 0):,}",
        "Tỷ lệ SPAM (%)": f"{(counts.get('spam', 0) / n_total * 100):.2f}%",
    })

df_splits = pd.DataFrame(split_summary)
display(df_splits)


Tổng số bản ghi thô: 43,849


,Chỉ tiêu kiểm định,Số lượng
0,Tổng số dòng đầu vào (Input Rows),"43,849"
1,Dòng nhãn không hợp lệ (Invalid Label),"2,195"
2,Dòng văn bản rỗng (Empty Text),"2,192"
3,Nhóm mâu thuẫn nhãn (Conflicting Groups),738
4,Số dòng mâu thuẫn nhãn (Conflicting Rows),"1,480"
5,Dòng trùng lặp loại bỏ (Duplicates Removed),"8,954"
6,Dòng sạch được giữ lại (Retained Clean Rows),"29,028"


Số bản ghi sạch giữ lại: 29,028


,Tập phân chia,Tổng số mẫu,Số lượng HAM,Tỷ lệ HAM (%),Số lượng SPAM,Tỷ lệ SPAM (%)
0,Train (80%),"23,223","13,309",57.31%,"9,914",42.69%
1,Val (10%),"2,903","1,664",57.32%,"1,239",42.68%
2,Test (10%),"2,902","1,663",57.31%,"1,239",42.69%


## 3. Tiền Xử Lý Văn Bản & Trích Xuất Đặc Trưng Cấu Trúc (SpamTextProcessor)
- Chuẩn hóa Unicode NFKD, lowercase.
- Xóa thẻ HTML bằng Parser tự viết.
- Token hóa URL (`urltoken`), số tiền (`moneytoken`), số (`numbertoken`).
- Loại bỏ stop words và dấu câu.
- Trích xuất đặc trưng cấu trúc: độ dài, chữ hoa, số lượng dấu `!`, `$`, `?`.


In [8]:
processor = SpamTextProcessor()

# Trích xuất và so sánh tiền xử lý trên 5 email mẫu dưới dạng DataFrame
sample_records = []
for i, row in train_df.head(5).reset_index(drop=True).iterrows():
    raw_text = str(row.get("raw_text", row.get("text", "")))
    cleaned = processor.normalize_text(raw_text)
    struct = processor.extract_structural_features(raw_text)
    sample_records.append({
        "Mẫu": f"Email #{i + 1}",
        "Nhãn": str(row["label"]).upper(),
        "Văn bản gốc (Raw Text)": raw_text[:75] + ("..." if len(raw_text) > 75 else ""),
        "Văn bản chuẩn hóa (Cleaned)": cleaned[:75] + ("..." if len(cleaned) > 75 else ""),
        "Tỷ lệ chữ in hoa": round(struct["uppercase_ratio"], 4),
        "Dấu '!'": int(struct["exclamation_count"]),
        "Dấu '$'": int(struct["dollar_count"]),
        "Có URL": bool(struct["has_url"]),
    })

df_samples = pd.DataFrame(sample_records)
display(df_samples)


,Mẫu,Nhãn,Văn bản gốc (Raw Text),Văn bản chuẩn hóa (Cleaned),Tỷ lệ chữ in hoa,Dấu '!',Dấu '$',Có URL
0,Email #1,HAM,Maybe you should find something else to do ins...,maybe find something else instead,0.0238,0,0,False
1,Email #2,HAM,How are you enjoying this semester? Take care ...,how enjoying semester take care brother,0.0455,0,0,False
2,Email #3,SPAM,escapenumberd escapenumberd escapenumberd esc...,escapenumberd escapenumberd escapenumberd esca...,0.0000,0,0,False
3,Email #4,SPAM,on escapenumber escapenumber escapenumber stev...,escapenumber escapenumber escapenumber steve b...,0.0000,0,0,False
4,Email #5,SPAM,on escapenumber escapenumber escapenumber paul...,escapenumber escapenumber escapenumber paul la...,0.0000,0,0,False


## 4. Vector Hóa Văn Bản Bằng TF-IDF From-Scratch (TfidfVectorizerScratch)
- Sublinear Term Frequency: $\text{TF} = 1 + \ln(\text{count})$.
- Smooth Inverse Document Frequency: $\text{IDF} = \ln((1 + N)/(1 + \text{DF})) + 1$.
- Chuẩn hóa vector $L_2$ norm.
- **Chỉ fit trên Train**, sau đó transform trên Val và Test.


In [9]:
# Chuẩn bị cột model_text
train_prepared = processor.prepare_feature_frame(train_df)
val_prepared = processor.prepare_feature_frame(val_df)
test_prepared = processor.prepare_feature_frame(test_df)

train_texts = [r["model_text"] for r in train_prepared]
val_texts = [r["model_text"] for r in val_prepared]

# Fit TF-IDF chỉ trên Train
tfidf = TfidfVectorizerScratch(min_df=2, max_features=3000, sublinear_tf=True, smooth_idf=True, norm="l2")
tfidf.fit(train_texts)

X_train_word = tfidf.transform(train_texts)
X_val_word = tfidf.transform(val_texts)

print(f"Số lượng từ vựng (Vocabulary size): {len(tfidf.vocabulary_):,}")

# Bảng đối sánh Top 10 từ IDF cao nhất (đặc trưng) và thấp nhất (phổ biến)
terms_by_idf = sorted(tfidf.vocabulary_.items(), key=lambda item: tfidf.idf_diag_[item[1]], reverse=True)
df_idf_analysis = pd.DataFrame({
    "Thứ hạng": range(1, 11),
    "Từ đặc trưng (Top IDF)": [t[0] for t in terms_by_idf[:10]],
    "Giá trị IDF cao": [round(tfidf.idf_diag_[t[1]], 4) for t in terms_by_idf[:10]],
    "Từ phổ biến nhất (Low IDF)": [t[0] for t in reversed(terms_by_idf[-10:])],
    "Giá trị IDF thấp": [round(tfidf.idf_diag_[t[1]], 4) for t in reversed(terms_by_idf[-10:])],
})
display(df_idf_analysis)

# Bảng kích thước ma trận từ vựng
df_matrix_shape = pd.DataFrame([
    {"Tập dữ liệu": "Train Word Matrix", "Kích thước (Shape)": str(X_train_word.shape), "Số phần tử khác 0 (NNZ)": f"{X_train_word.nnz:,}"},
    {"Tập dữ liệu": "Val Word Matrix", "Kích thước (Shape)": str(X_val_word.shape), "Số phần tử khác 0 (NNZ)": f"{X_val_word.nnz:,}"},
])
display(df_matrix_shape)


Số lượng từ vựng (Vocabulary size): 3,000


,Thứ hạng,Từ đặc trưng (Top IDF),Giá trị IDF cao,Từ phổ biến nhất (Low IDF),Giá trị IDF thấp
0,1,affected,6.308,escapenumber,1.7411
1,2,agenda,6.308,http,2.2003
2,3,blog,6.308,com,2.3335
3,4,closer,6.308,please,2.3438
4,5,contacting,6.308,not,2.3764
5,6,ended,6.308,numbertoken,2.6460
6,7,ending,6.308,one,2.6675
7,8,equivalent,6.308,e,2.6764
8,9,escapenumberc,6.308,get,2.6922
9,10,escapenumbert,6.308,time,2.7337


,Tập dữ liệu,Kích thước (Shape),Số phần tử khác 0 (NNZ)
0,Train Word Matrix,"(23223, 3000)","1,247,980"
1,Val Word Matrix,"(2903, 3000)","157,204"


## 5. Xây Dựng Ma Trận Đặc Trưng Lai Toàn Diện & Đối Sánh Các Tổ Hợp Đặc Trưng
- Ghép nối đa khối: `[Word TF-IDF + Bigrams + Char N-grams + Keywords + Scaled Numerics]`.
- Đối sánh 5 tổ hợp đặc trưng theo yêu cầu đề bài:
  1. `word_tfidf`: Chỉ dùng TF-IDF từ đơn.
  2. `word_char_frequency`: Word TF-IDF + tần suất ký tự đặc biệt (`!`, `$`).
  3. `word_char_ngrams`: Word TF-IDF + Character n-grams (3-5 ký tự).
  4. `word_keywords`: Word TF-IDF + 10 nhóm từ khóa spam có trọng số.
  5. `hybrid`: Kết hợp toàn bộ các nguồn đặc trưng.


In [10]:
config = SpamExperimentConfig()
signal_extractor = SpamSignalFeatureExtractor(config)
builder = HybridFeatureBuilderScratch(config, processor, signal_extractor)

# Fit trên Train và Transform trên Val & Test
X_train_hybrid = builder.fit_transform(train_df)
X_val_hybrid = builder.transform(val_df)
X_test_hybrid = builder.transform(test_df)

print(f"Tổng số đặc trưng lai: {X_train_hybrid.shape[1]:,} chiều")

# Bảng phân đoạn các khối đặc trưng
df_slices = pd.DataFrame([
    {"Khối đặc trưng": k, "Chỉ số bắt đầu": v[0], "Chỉ số kết thúc": v[1], "Số chiều": v[1] - v[0]}
    for k, v in builder.feature_slices_.items()
])
display(df_slices)

# Thử nghiệm 5 tổ hợp đặc trưng
combinations = experiment_feature_combinations(train_df, val_df)
summary_rows = []
for name, (xt, xv) in combinations.items():
    summary_rows.append({
        "Tổ hợp đặc trưng": name,
        "Số chiều (Features)": xt.shape[1],
        "Train Shape": str(xt.shape),
        "Val Shape": str(xv.shape),
        "NNZ (Train non-zeros)": xt.nnz,
        "Độ thưa (%)": round(100.0 * (1.0 - xt.nnz / (xt.shape[0] * xt.shape[1])), 4),
    })

df_combos = pd.DataFrame(summary_rows)
display(df_combos)


Tổng số đặc trưng lai: 4,318 chiều


,Khối đặc trưng,Chỉ số bắt đầu,Chỉ số kết thúc,Số chiều
0,word,0,3000,3000
1,bigram,3000,3800,800
2,char,3800,4300,500
3,keyword,4300,4310,10
4,numeric,4310,4318,8


,Tổ hợp đặc trưng,Số chiều (Features),Train Shape,Val Shape,NNZ (Train non-zeros),Độ thưa (%)
0,word_tfidf,3000,"(23223, 3000)","(2903, 3000)",1207544,98.2667
1,word_char_frequency,3002,"(23223, 3002)","(2903, 3002)",1210646,98.2634
2,word_char_ngrams,3500,"(23223, 3500)","(2903, 3500)",5257097,93.5322
3,word_keywords,3010,"(23223, 3010)","(2903, 3010)",1226863,98.2449
4,hybrid,4318,"(23223, 4318)","(2903, 4318)",5527789,94.4875


## 6. Lựa Chọn Đặc Trưng (Feature Selection) & Chuẩn Hóa MaxAbsScaler From-Scratch
- **Kiểm định thống kê Chi-Square ($\chi^2$)** & Feature Activity & Naive Bayes Difference để xếp hạng độ phân biệt của từng đặc trưng.
- **Quét chọn dải Top-K đặc trưng (`sweep_top_k`)**: Cân bằng F1-score trên Validation, tối ưu hóa tỷ lệ tiết kiệm số chiều và kiểm soát overfit.
- **Chuẩn hóa `MaxAbsScalerScratch`**: Bảo toàn tính không âm ($X \ge 0$) và cấu trúc ma trận thưa mà không gây tràn bộ nhớ RAM.


In [11]:
from src.features.feature_engineering import ManualFeatureSelectorScratch, MaxAbsScalerScratch
from src.evaluation.visualization import plot_top_k_feature_sweep

# 1. Trích xuất nhãn nhị phân cho Train và Val từ DataFrame
y_train = (train_df['label'] == 'spam').astype(int).values
y_val = (val_df['label'] == 'spam').astype(int).values

# 2. Xếp hạng độ quan trọng kết hợp (Combined Importance Score)
selector = ManualFeatureSelectorScratch()
combined_scores, ranked_indices = selector.compute_combined_importance(X_train_hybrid, y_train)
print(f"Đã xếp hạng toàn bộ {len(ranked_indices):,} đặc trưng theo điểm kết hợp.")

# 3. Quét dải Top-K ứng viên để tìm điểm cân bằng F1 vs Số chiều
candidate_ks = [500, 1000, 1500, 2000, 2500, min(3000, X_train_hybrid.shape[1]), X_train_hybrid.shape[1]]
best_sweep, sweep_df = selector.sweep_top_k(
    X_train_hybrid, y_train, X_val_hybrid, y_val, ranked_indices, candidate_k_list=candidate_ks, recall_target=0.85
)

# Hiển thị bảng đối sánh chi tiết các mốc Top-K
display(sweep_df)

print(f"Top-K tối ưu được chọn: {best_sweep['top_k']} đặc trưng (Tiết kiệm {best_sweep['dimension_saving_pct']:.1f}% số chiều, Val F1 = {best_sweep['val_f1']:.4f})")
plot_top_k_feature_sweep(sweep_df)

# 4. Chuẩn hóa MaxAbsScalerScratch trên ma trận thưa
scaler = MaxAbsScalerScratch()
X_train_scaled = scaler.fit_transform(X_train_hybrid)
print(f"Chuẩn hóa MaxAbs hoàn tất. Shape: {X_train_scaled.shape}, Non-zeros: {X_train_scaled.nnz}")


Đã xếp hạng toàn bộ 4,318 đặc trưng theo điểm kết hợp.


,top_k,dimension_saving_pct,val_accuracy,val_precision,val_recall,val_f1,train_val_gap,target_met,balance_score,selected_indices
0,500,88.420565,0.677919,0.636691,0.571429,0.602297,0.019708,False,0.675293,"[3130, 4311, 3003, 4303, 3005, 4316, 4310, 430..."
1,1000,76.841130,0.681020,0.632291,0.603713,0.617671,0.020225,False,0.659879,"[3130, 4311, 3003, 4303, 3005, 4316, 4310, 430..."
2,1500,65.261695,0.688598,0.638315,0.623890,0.631020,0.016953,False,0.647016,"[3130, 4311, 3003, 4303, 3005, 4316, 4310, 430..."
3,2000,53.682260,0.695832,0.643780,0.643261,0.643520,0.012561,False,0.634210,"[3130, 4311, 3003, 4303, 3005, 4316, 4310, 430..."
4,2500,42.102825,0.693765,0.640675,0.643261,0.641965,0.018288,False,0.606764,"[3130, 4311, 3003, 4303, 3005, 4316, 4310, 430..."
5,3000,30.523390,0.693421,0.639712,0.644875,0.642283,0.017728,False,0.583907,"[3130, 4311, 3003, 4303, 3005, 4316, 4310, 430..."
6,4318,0.000000,0.696176,0.643373,0.646489,0.644928,0.015661,False,0.526049,"[3130, 4311, 3003, 4303, 3005, 4316, 4310, 430..."


Top-K tối ưu được chọn: 4318 đặc trưng (Tiết kiệm 0.0% số chiều, Val F1 = 0.6449)
Chuẩn hóa MaxAbs hoàn tất. Shape: (23223, 4318), Non-zeros: 5527789


## 7. Xuất Dữ Liệu Đệm (Cache Persistence) Vào data/processed/
- Lưu trữ các tập phân tầng sạch (`train_clean.csv`, `val_clean.csv`, `test_clean.csv`).
- Lưu trữ các ma trận đặc trưng CSR (`X_train.npz`, `X_val.npz`, `X_test.npz`) và vector nhãn `y`.
- Giúp Task 2 nạp ngay tức thì mà không phải lặp lại tiền xử lý tốn kém.


In [12]:
export_result = export_processed_data(
    train_df, val_df, test_df, builder=builder
)

# Bảng trạng thái xuất bộ đệm Task 1
df_export_status = pd.DataFrame([
    {"Tài nguyên xuất": k, "Đường dẫn / Trạng thái": str(v)}
    for k, v in export_result.items()
])
display(df_export_status)

print("Đã bàn giao đầy đủ đầu ra của Task 1 cho Task 2 thành công!")


,Tài nguyên xuất,Đường dẫn / Trạng thái
0,train_csv,E:\Hoc_May_Tren_truong\lab_1 spam_classifier\d...
1,val_csv,E:\Hoc_May_Tren_truong\lab_1 spam_classifier\d...
2,test_csv,E:\Hoc_May_Tren_truong\lab_1 spam_classifier\d...
3,matrices_saved,True


Đã bàn giao đầy đủ đầu ra của Task 1 cho Task 2 thành công!
